# درس 11 - پروتکل زمینه مدل (MCP)

پروتکل زمینه مدل **(MCP)** یک استاندارد باز است که به عامل‌ها امکان می‌دهد در زمان اجرا به‌صورت پویا ابزارها، منابع و منابع داده را کشف و استفاده کنند. به‌جای هاردکد کردن ابزارها در یک عامل، MCP به عامل‌ها اجازه می‌دهد به سرورهای خارجی متصل شوند که قابلیت‌ها را بر اساس درخواست آشکار می‌کنند.

در این درس، شما یاد خواهید گرفت:
- MCP چیست و چرا برای سامانه‌های عامل اهمیت دارد
- معماری کلاینت-سرور MCP چگونه کار می‌کند
- چگونه عامل‌هایی بسازید که از کشف ابزار به سبک MCP استفاده می‌کنند


## راه‌اندازی

**پیش‌نیازها:**
- پروژه Azure AI Foundry با یک مدل مستقر
- برای احراز هویت دستور `az login` را اجرا کنید


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## پروتکل زمینه مدل (MCP) چیست؟

MCP یک روش استاندارد برای کشف و تعامل عوامل هوش مصنوعی با ابزارها و منابع داده خارجی تعریف می‌کند:

- **MCP Server**: ابزارها، منابع و پرومپت‌ها را از طریق یک پروتکل استاندارد ارائه می‌دهد
- **MCP Client**: محیط اجرای عامل که به سرورها متصل می‌شود و قابلیت‌های در دسترس را کشف می‌کند
- **Dynamic Discovery**: عوامل نیازی به ابزارهای کدگذاری‌شده ثابت ندارند — آنها در زمان اجرا آنچه در دسترس است را کشف می‌کنند

این برای ساخت سیستم‌های عاملی توسعه‌پذیر قدرتمند است که در آنها می‌توان قابلیت‌های جدید را بدون تغییر کد عامل اضافه کرد.


## نحوه کار MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. عامل (کلاینت MCP) به یک سرور MCP متصل می‌شود
2. سرور با فهرستی از ابزارهای در دسترس و شِماهایشان پاسخ می‌دهد
3. عامل سپس می‌تواند هر ابزار کشف‌شده‌ای را در حین استدلال خود فراخوانی کند
4. نتایج از طریق همان پروتکل بازمی‌گردند


## شبیه‌سازی کشف ابزارهای MCP

از آنجا که یک سرور واقعی MCP به یک فرایند سرور در حال اجرا نیاز دارد، الگو را با استفاده از توابع `@tool` نشان خواهیم داد که شبیه‌سازی می‌کنند آنچه یک سرویس اقامتی متصل به MCP ارائه می‌دهد.

در محیط تولید، این ابزارها به‌صورت پویا از یک سرور MCP کشف می‌شدند، نه اینکه به‌صورت محلی تعریف شوند.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## ساخت یک عامل با ابزارهای به سبک MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP در محیط تولید

در محیط تولید، MCP الگوهای قدرتمندی را فراهم می‌کند:

- **کشف ابزارهای پویا**: عامل‌ها به سرورهای MCP متصل می‌شوند و ابزارها را در زمان اجرا کشف می‌کنند
- **معماری تفکیک‌شده**: ارائه‌دهندگان ابزار می‌توانند به‌طور مستقل از عامل به‌روزرسانی شوند
- **اشتراک‌گذاری بین‌سازمانی**: تیم‌ها می‌توانند قابلیت‌ها را از طریق سرورهای MCP در دسترس قرار دهند تا هر عاملی بتواند از آن‌ها استفاده کند
- **پشتیبانی Microsoft Agent Framework**: MAF به‌صورت داخلی از کلاینت MCP از طریق ادغام `mcp` پشتیبانی می‌کند

برای استفاده از یک سرور MCP واقعی با MAF، از طریق `hosted_mcp_tool()` یا ادغام کلاینت MCP متصل می‌شوید.

**بیشتر بدانید:**
- [مشخصات MCP](https://modelcontextprotocol.io/)
- [پشتیبانی MCP در Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## خلاصه

در این درس آموختید:
- **MCP** یک استاندارد باز برای کشف پویا ابزارها بین عامل‌ها و ارائه‌دهندگان ابزار است
- معماری **کلاینت-سرور** به عامل‌ها اجازه می‌دهد در زمان اجرا قابلیت‌ها را کشف کنند
- MCP امکان ایجاد **سیستم‌های عاملی قابل توسعه و جداشده** را فراهم می‌کند که در آن‌ها می‌توان ابزارها را بدون تغییر در کد اضافه کرد
- Microsoft Agent Framework پشتیبانی **داخلی از MCP** را برای استفاده در محیط تولید فراهم می‌کند


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
سلب‌مسئولیت:
این سند با استفاده از سرویس ترجمهٔ هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در پی دقت هستیم، لطفاً به‌خاطر داشته باشید که ترجمه‌های خودکار ممکن است حاوی خطاها یا نادرستی‌هایی باشند. سند اصلی به زبانِ اصلی خود باید به‌عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، توصیه می‌شود از ترجمهٔ حرفه‌ای انسانی استفاده شود. ما در برابر هرگونه سوتفاهم یا تفسیر نادرستی که ناشی از استفاده از این ترجمه باشد، مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
